### The Baselines

* **Random Classifier:**
* **Heuristic:**
* **Multi-Class Linear Classifier:** 

### Neural Network

* **Basic Neural Network / Multilayer Perceptron:**
* **Regularized NN:**

### Natural Language Processing:

* **Bag of Words Model:**
* **RNN Sentiment Analysis:**

### Extra

* **Attention Models:** Use attention mechanisms on the titles/descriptions
* **Some Fancy Pretrained Model maybe BERT too**
* **Just run it into an LLM and see how it does**


In [27]:
# access parquet :D files with pandas
import pandas as pd
import polars as pl

df = pl.read_parquet('data/youtube_data.parquet')
df

title,channel_name,daily_rank,daily_movement,weekly_movement,snapshot_date,country,view_count,like_count,comment_count,description,video_id,channel_id,video_tags,kind,publish_date,langauge,like_tier
str,cat,u8,i16,i16,"datetime[μs, UTC]",cat,i64,i32,i32,str,str,str,str,cat,"datetime[μs, UTC]",cat,cat
"""Nyasha David - Tsvodi (Officia…","""Nyasha David""",1,0,15,2026-02-18 00:00:00 UTC,"""ZW""",430573,13860,1139,"""#Tsvodi #nyashadavid #mafindif…","""07pinMbPq2Q""","""UCx1LPxEtJWIpvtXTYadAdKg""","""""","""youtube#video""",2026-02-09 00:00:00 UTC,"""und""","""pretty_viral"""
"""WAR MACHINE | Official Trailer…","""Netflix""",2,0,0,2026-02-18 00:00:00 UTC,"""ZW""",12531660,191591,15336,"""During the final stage of U.S.…","""AFuE1LRxm80""","""UCWOA1ZGywLbqmigxE4Qlvuw""","""Alan Ritchson, Alan Ritchson W…","""youtube#video""",2026-02-04 00:00:00 UTC,"""en-US""","""really_viral"""
"""10 Seconds vs 1 Hour MODERN MO…","""Cash""",3,0,47,2026-02-18 00:00:00 UTC,"""ZW""",491455,6139,1656,"""Instagram: https://www.instagr…","""EbBjlznDwzk""","""UC0eLBYhxW9HC0P9PXQ73mpQ""","""Cash, Nico, Nico and Cash, Cas…","""youtube#video""",2026-02-16 00:00:00 UTC,"""en-US""","""a_little_viral"""
"""Master H - Impossible (Officia…","""Master H""",4,0,3,2026-02-18 00:00:00 UTC,"""ZW""",532388,14552,1281,"""Artist : Master H Song : Impos…","""TIG5IzusQ24""","""UC5jrU8WxzE16gPTnlqQAqew""","""""","""youtube#video""",2026-02-06 00:00:00 UTC,"""en""","""pretty_viral"""
"""“Spider-Noir” – Authentic Blac…","""Prime Video""",5,0,45,2026-02-18 00:00:00 UTC,"""ZW""",13406903,239798,10614,"""With no power comes no respons…","""HgMbkitzhEM""","""UCQJWtTnAHhEG5w4uN0udnUQ""","""spider noir official teaser, s…","""youtube#video""",2026-02-12 00:00:00 UTC,"""en""","""really_viral"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""انهيار الطبيبة هديل الجمل عند …","""شبكة قدس الاخبارية""",46,4,4,2023-10-26 00:00:00 UTC,"""AE""",1064501,8823,815,"""تابعوا #شبكة_قدس عبر المنصات ا…","""OcPejiPT07U""","""UCFFuMGTQXLitNuiU4SIe5vQ""","""فلسطين, القدس, غزة, تاريخ فلسط…","""youtube#video""",2023-10-22 15:04:53 UTC,"""ar""","""a_little_viral"""
"""بدهم ايانا نترك بيتنا !""","""وليد ونور | Waleed & Noor""",47,3,3,2023-10-26 00:00:00 UTC,"""AE""",413648,36218,2070,"""حساباتنا على مواقع التواصل الإ…","""qX9XD-5ZGIE""","""UCHywC_HXMWAM6-XvjY8gnUw""","""وليد ونور, نور غسان, وليدمقداد…","""youtube#video""",2023-10-22 16:10:45 UTC,"""ar""","""pretty_viral"""
"""""إسرائيل"" تنزف!.. هل ينهار الا…","""المخبر الاقتصادي - Mokhbir Eqt…",48,2,2,2023-10-26 00:00:00 UTC,"""AE""",2076478,197026,12930,"""لو انت شركة أو تاجر اتواصل مع …","""s7n2wzIWjtY""","""UC4kRorAXuIkyIX6vwXKaLWg""","""المخبر الاقتصادي, اخبار الاقتص…","""youtube#video""",2023-10-19 17:00:08 UTC,"""ar""","""really_viral"""


In [28]:
# Dedpulicate videos
df = df.sort("snapshot_date")
df_unique = df.unique(subset=["video_id"], keep="last")

# Sort the new unique dataframe by when the videos were published
df_unique = df_unique.sort("publish_date")

# Split by cutoff date
cutoff_date = df_unique['publish_date'].quantile(0.8, "lower")
print(f"Using publish_date {cutoff_date} as the cutoff point.\n")

train_df = df_unique.filter(pl.col("publish_date") <= cutoff_date)
test_df = df_unique.filter(pl.col("publish_date") > cutoff_date)

X_train = train_df.drop("like_tier")
y_train = train_df.get_column("like_tier")

X_test = test_df.drop("like_tier")
y_test = test_df.get_column("like_tier")

print(f"Training set size: {len(X_train)} unique videos (past)")
print(f"Testing set size:  {len(X_test)} unique videos (future)")

Using publish_date 2025-11-05 00:00:00+00:00 as the cutoff point.

Training set size: 366469 unique videos (past)
Testing set size:  90674 unique videos (future)


In [29]:
# a random classifier that takes account in class distributions
import numpy as np
from sklearn.metrics import classification_report

# Calculate class probabilities directly from the y_train Series
counts_df = y_train.value_counts()
labels = counts_df.get_column("like_tier").to_list()
probs = (counts_df.get_column("count") / len(y_train)).to_list()

random_preds = np.random.choice(
    labels,
    size=len(y_test),
    p=probs
)

print("Random Classifier Evaluation Report:")
print(classification_report(y_test, random_preds, zero_division=0))

Random Classifier Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.81      0.49      0.61     73784
  pretty_viral       0.16      0.32      0.21     14234
  really_viral       0.03      0.13      0.04      2486
   super_viral       0.00      0.05      0.00       170

      accuracy                           0.45     90674
     macro avg       0.25      0.25      0.22     90674
  weighted avg       0.69      0.45      0.53     90674



In [30]:
# find the most common class in the training labels
majority_class = y_train.mode()[0]

# predict that class for every test sample
heuristic_preds = [majority_class] * len(y_test)


print("Majority Class:", majority_class)
print("Huristic Classifier Evaluation Report:")
print(classification_report(y_test, heuristic_preds, zero_division=0))

Majority Class: a_little_viral
Huristic Classifier Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.81      1.00      0.90     73784
  pretty_viral       0.00      0.00      0.00     14234
  really_viral       0.00      0.00      0.00      2486
   super_viral       0.00      0.00      0.00       170

      accuracy                           0.81     90674
     macro avg       0.20      0.25      0.22     90674
  weighted avg       0.66      0.81      0.73     90674



This suggests the data is relatively balanced since we are getting an accuracy of around 25%! So there isn't one specific class that dominates over the rest which is good.

In [36]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# multi-class linear classifier

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
EPOCHS = 10

# numeric only first not like_count lmao
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.select(numeric_features).to_numpy())
X_test_scaled = scaler.transform(X_test.select(numeric_features).to_numpy())

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

# Convert everything to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

# Create Dataloader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

# Simple linear classifier
model = nn.Linear(len(numeric_features), len(label_map)).to(device)
criterion = nn.CrossEntropyLoss()
# We'll use SGD to make the baseline less powerful Adam does too much for us
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

print("Training Linear Classifier...")
for epoch in range(EPOCHS):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    
    # Move predictions back to CPU for Scikit-Learn
    predicted_np = predicted.cpu().numpy()

print("\nLinear Classifier Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))

Using device: mps
Training Linear Classifier...

Linear Classifier Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.85      0.98      0.91     73784
  pretty_viral       0.47      0.17      0.25     14234
  really_viral       0.74      0.18      0.29      2486
   super_viral       0.57      0.45      0.50       170

      accuracy                           0.83     90674
     macro avg       0.66      0.44      0.49     90674
  weighted avg       0.79      0.83      0.79     90674

